# Text-to-SQL agent on a parquet-flavored chDB store

A working LangGraph agent that answers natural-language questions against an in-process chDB engine via `langchain-chdb[sql]` + `SQLDatabaseToolkit`. Designed to fit in ~30 lines of actual logic; everything else is the seed data.

## Requirements

```bash
pip install "langchain-chdb[sql]" langchain-anthropic langgraph
export ANTHROPIC_API_KEY=...
```

## Plumbing

1. Build a chDB SQLAlchemy engine via the `chdb-sqlalchemy` dialect (pulled by the `[sql]` extra).
2. Seed a small `orders` table — replace with `SELECT * FROM file('orders.parquet', 'Parquet')` or `s3('s3://bucket/orders.parquet')` for real data.
3. Wrap the engine in `langchain_community.utilities.SQLDatabase`.
4. Give the agent the standard `SQLDatabaseToolkit` tools (list tables, describe schema, run SQL).
5. Drive everything from a LangGraph `create_react_agent` so the agent picks its own tool order.

Federation is one substitution away: `engine = create_engine("chdb:///./local.chdb")` then write SQL that joins the local cache with `remoteSecure('your-cluster.clickhouse.cloud:9440', 'db.tbl', ...)`.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from langgraph.prebuilt import create_react_agent
from sqlalchemy import create_engine, text

# 1. chDB engine — in-memory for this demo; swap in a file path for persistence.
engine = create_engine("chdb:///:memory:")

# 2. Seed sample data. Real workloads would point chDB at a parquet / S3 source.
with engine.begin() as conn:
    conn.execute(text(
        "CREATE TABLE orders ("
        "  order_id Int32, customer String, product String, "
        "  quantity Int32, unit_price Decimal(10,2), order_date Date"
        ") ENGINE=Memory"
    ))
    conn.execute(text(
        "INSERT INTO orders VALUES "
        "(1,'Alice','Widget A',3,12.50,'2026-04-15'),"
        "(2,'Bob','Widget B',5,8.00,'2026-04-16'),"
        "(3,'Alice','Widget A',2,12.50,'2026-04-18'),"
        "(4,'Carol','Gadget C',1,99.99,'2026-04-22'),"
        "(5,'Bob','Widget A',4,12.50,'2026-05-01')"
    ))

# 3. Wrap engine for LangChain.
db = SQLDatabase(engine)

# 4. Build agent: Claude + the standard SQL toolkit.
llm = ChatAnthropic(model="claude-sonnet-4-5")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
agent = create_react_agent(llm, toolkit.get_tools())

# 5. Ask a question in plain English.
result = agent.invoke({
    "messages": [("user", "Who is our top customer by total revenue? Show the figure.")]
})
print(result["messages"][-1].content)

## Expected behavior

The agent typically follows this trajectory:

1. `sql_db_list_tables` → sees `orders`.
2. `sql_db_schema` → reads the column types.
3. `sql_db_query` → runs something close to:

   ```sql
   SELECT customer, sum(quantity * unit_price) AS revenue
   FROM orders
   GROUP BY customer
   ORDER BY revenue DESC
   LIMIT 1;
   ```

4. Returns a sentence naming Bob and the revenue figure.

The dialect under the hood is [`chdb-sqlalchemy`](https://github.com/chdb-io/chdb-sqlalchemy) — it handles table reflection, type mapping (including the `Decimal(10,2)` shown here), and the `info`-shape introspection that `SQLDatabaseToolkit` consumes.

## Next steps

- **Persist the database.** Swap `chdb:///:memory:` for `chdb:////tmp/my.chdb`.
- **Read from S3 / parquet / a URL directly.** chDB's table functions work inside a `CREATE TABLE ... AS SELECT * FROM file('orders.parquet', 'Parquet')` or via dynamic queries against `s3('s3://bucket/key')`, `url('https://...')`, `iceberg(...)`, etc.
- **Federation.** Join the local cache with a ClickHouse Cloud cluster using `remoteSecure('host:9440', 'db.tbl', 'user', 'password')` in the same SQL the agent emits.
- **Vector search.** Combine `ChDBVectorStore` for retrieval with this `SQLDatabaseToolkit` path for structured queries — both run inside the same chDB process, so a single query can `JOIN` retrieved-by-similarity docs against analytical aggregates.